# First steps in Python

This is the beginning of the **core path** through Part I. Every code cell here
runs on its own — there is nothing to install beyond what the [setup page](#)
covered, and we use only the Python language itself, no external libraries yet.
Work through the cells top to bottom, changing values and re-running to see what
happens. That experimentation *is* the lesson.

:::{admonition} What you will be able to do by the end
:class: tip
- store and label values with variables, and recognise Python's basic types
- do arithmetic, including a unit conversion in SI units
- build reports with f-strings
- collect many values in a list and step through them with a loop
- make decisions with `if`/`elif`/`else`
- package reusable logic into a function

Estimated time: 30–45 minutes.
:::

:::{admonition} How this book is laid out
:class: note
The main flow is the core path: read it straight through and you will not miss
anything essential. Collapsible boxes labelled **"Going deeper"** hold optional
material for readers who want the why behind the how — open them if you are
curious, skip them with no loss of continuity.
:::


## A running example

Throughout Part I we keep returning to one quantity: the near-surface air
temperature. Starting from something physical keeps the code meaningful rather
than abstract. For now we use a few **approximate** monthly mean values for
Lausanne, in degrees Celsius — just enough to compute with. Later we will load
the real, full dataset.


## Variables and types

A *variable* is a name attached to a value. You create one with `=` (read it as
"is assigned", not "equals"):


In [1]:
city = "Lausanne"
july_temp = 19.5    # approximate mean July 2 m air temperature, in degrees Celsius
print(city, "->", july_temp, "deg C")

Lausanne -> 19.5 deg C


Every value has a *type*. The three you will meet constantly are `str` (text),
`float` (a real number), and `int` (a whole number). A fourth, `bool`, is either
`True` or `False`.


In [2]:
print(type(city))
print(type(july_temp))
print(type(2025))

<class 'str'>
<class 'float'>
<class 'int'>


:::{admonition} Going deeper: why `19.5` and `19` are different types
:class: dropdown
`july_temp` is a `float` because it has a decimal point; `2025` is an `int`.
The distinction matters because computers store them differently and floats
carry a tiny, finite precision. A classic surprise:

```python
0.1 + 0.2   # -> 0.30000000000000004, not exactly 0.3
```

This is not a Python bug; it is how binary floating-point represents decimals,
and it is shared by essentially every programming language. We will care about
it when comparing computed temperatures for equality later in Part I.
:::


## Arithmetic, and a first taste of units

Python's arithmetic operators are `+ - * /` plus `**` for powers. We will use
them to convert our temperature from degrees Celsius to **kelvin**, the SI unit
of temperature. The conversion is $T_\mathrm{K} = T_{^\circ\mathrm{C}} + 273.15$:


In [3]:
july_temp_K = july_temp + 273.15
july_temp_K

292.65

Keeping unit conversions explicit in the code — rather than hiding them — is a
habit worth forming early; a large fraction of scientific software bugs are unit
errors.


In [4]:
# a few more operators
print(2 ** 10)     # power: 2 to the 10th
print(17 / 5)      # true division -> float
print(17 // 5)     # floor division -> whole part
print(17 % 5)      # modulo -> remainder

1024
3.4
3
2


:::{admonition} Going deeper: integer vs floating division
:class: dropdown
`/` always gives a `float`, even for `4 / 2` (which is `2.0`). `//` discards the
fractional part, and `%` returns what is left over. The pair is handy for
cyclic quantities — for example, mapping any month number onto 0–11, or any hour
onto a 24-hour clock with `hour % 24`.
:::


## Reporting results with f-strings

To build a sentence from values, put an `f` before the quotes and write
expressions inside `{}`:


In [5]:
print(f"The mean July temperature in {city} is {july_temp} deg C ({july_temp_K} K).")

The mean July temperature in Lausanne is 19.5 deg C (292.65 K).


:::{admonition} Going deeper: controlling the format
:class: dropdown
You can format numbers inside the braces. A colon introduces a *format spec*:
`{value:.1f}` shows one decimal place, `{value:.0f}` rounds to an integer.

```python
print(f"{july_temp_K:.2f} K")   # -> 292.65 K
```

This keeps printed precision honest — reporting `292.6500000001 K` would imply
accuracy you do not have.
:::


## Many values at once: lists

A single temperature is not very interesting. A *list* holds an ordered
collection, written with square brackets:


In [6]:
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

temps_c = [1.5, 2.5, 6.0, 9.5, 14.0, 17.5,
           19.5, 19.0, 15.0, 10.5, 5.0, 2.5]   # approximate, deg C

len(temps_c)   # how many values?

12

You reach individual elements by their *index*. Python counts from **0**, and
negative indices count from the end:


In [7]:
print(temps_c[0])    # first value (January)
print(temps_c[-1])   # last value (December)
print(temps_c[5:8])  # a slice: June, July, August

1.5
2.5
[17.5, 19.5, 19.0]


The slice `5:8` includes index 5 but stops *before* index 8 — a convention that
feels odd at first and becomes second nature.

:::{admonition} Going deeper: lists can be changed in place
:class: dropdown
Lists are *mutable* — you can reassign an element:

```python
temps_c[0] = 1.7   # correct January
```

This power has a sharp edge: if two names point at the same list, changing one
changes "both", because they are the same object. When you need a collection
that cannot be modified, use a *tuple* (round brackets). We will rely on
immutability when we want axis labels that must not drift.
:::


## Doing something to every value: loops

A `for` loop visits each element in turn. Here we compute the annual mean the
long way — accumulating a running total — so the operation is fully visible:


In [8]:
total = 0.0
for t in temps_c:
    total += t        # add each value to the running total

mean_c = total / len(temps_c)
print(f"Annual mean: {mean_c:.1f} deg C")

Annual mean: 10.2 deg C


:::{admonition} Going deeper: looping over two lists together
:class: dropdown
`zip` pairs lists element by element, and `enumerate` hands you the index
alongside the value:

```python
for month, t in zip(months, temps_c):
    print(month, t)
```

We use `zip` in the next section to label each temperature with its month.
:::


## Making decisions: `if` / `elif` / `else`

Conditionals run different code depending on a test. Let us label each month as
cold, mild, or warm:


In [9]:
for month, t in zip(months, temps_c):
    if t >= 15:
        label = "warm"
    elif t >= 5:
        label = "mild"
    else:
        label = "cold"
    print(f"{month}: {t:>4} deg C  ->  {label}")

Jan:  1.5 deg C  ->  cold
Feb:  2.5 deg C  ->  cold
Mar:  6.0 deg C  ->  mild
Apr:  9.5 deg C  ->  mild
May: 14.0 deg C  ->  mild
Jun: 17.5 deg C  ->  warm
Jul: 19.5 deg C  ->  warm
Aug: 19.0 deg C  ->  warm
Sep: 15.0 deg C  ->  warm
Oct: 10.5 deg C  ->  mild
Nov:  5.0 deg C  ->  mild
Dec:  2.5 deg C  ->  cold


The tests are checked in order; the first true branch wins, and `else` catches
everything left over. The `>4` inside the f-string just right-aligns the number
in a field four characters wide, to line the output up.


## Packaging logic: functions

When you find yourself repeating an operation, wrap it in a *function*. A
function takes inputs, does work, and `return`s a result:


In [10]:
def to_kelvin(temp_c):
    """Convert a temperature from degrees Celsius to kelvin."""
    return temp_c + 273.15


def mean(values):
    """Return the arithmetic mean of a list of numbers."""
    return sum(values) / len(values)


annual_mean_c = mean(temps_c)
print(f"{annual_mean_c:.1f} deg C  =  {to_kelvin(annual_mean_c):.1f} K")

10.2 deg C  =  283.4 K


We rebuilt `mean` by hand to see what it does; in practice `sum` already exists,
and very soon `numpy` will compute means over *millions* of values in a single
fast call. Writing it once yourself first makes that shortcut meaningful rather
than magical.

:::{admonition} Going deeper: converting a whole list at once
:class: dropdown
A *list comprehension* builds a new list from an old one in one expression —
read it as "`to_kelvin(t)` for each `t` in `temps_c`":

```python
temps_k = [to_kelvin(t) for t in temps_c]
```

This is the conceptual seed of *vectorisation*: describing an operation over a
whole collection at once instead of writing the loop yourself. `numpy` turns the
same idea into something both shorter and far faster.
:::


## Labelling values: dictionaries

A list keeps values in order; a *dictionary* keeps them under named keys. Pairing
months with temperatures lets us look one up by name:


In [11]:
climatology = dict(zip(months, temps_c))
climatology["Jul"]

19.5

This key-value idea returns in a more powerful form with `xarray`, where instead
of `"Jul"` we will index a temperature field by labelled coordinates like
`time`, `latitude`, and `longitude` — and never count raw positions again.


## Exercises

Try these before moving on. Each has a hidden solution — attempt it first.

**1.** Write a function `annual_range(values)` that returns the difference
between the warmest and coldest months. (Hint: `max` and `min` exist.)

:::{admonition} Solution
:class: dropdown
```python
def annual_range(values):
    # peak-to-peak temperature range
    return max(values) - min(values)

annual_range(temps_c)   # -> 18.0
```
:::

**2.** Count how many months are at or below freezing (0 deg C). Print the count.

:::{admonition} Solution
:class: dropdown
```python
freezing = 0
for t in temps_c:
    if t <= 0:
        freezing += 1
print(freezing, "month(s) at or below freezing")
```
With these illustrative values the answer is 0 — adjust a value and re-run to
check your logic responds.
:::


## Where this is going

You can now store, transform, and summarise numbers in plain Python. That is
enough to express real computations — but doing it element by element does not
scale to a temperature field with millions of grid points. In the next chapter
we meet `numpy`, which replaces these hand-written loops with fast operations on
whole arrays, and shortly after that `xarray`, which attaches physical labels
(time, latitude, longitude) to those arrays so the code reads like the science.
